# Notebook 04 ? Reliability-Aware Agentic RAG System

This notebook demonstrates the runnable reliability layer added on top of the baseline RAG system from Notebook 02. It focuses on the implemented mechanisms and on observable behaviour in example failure cases.

Mechanisms covered here:

| Mechanism | Component | Role |
|---|---|---|
| **A** | EvidenceAnalyst | Pre-synthesis evidence gate |
| **G** | RecoveryAgent | Adaptive recovery (rewrite, strategy switch, PRF) |
| **E** | AbstentionGate | Unified abstention handler |
| **B** | GroundednessVerifier | Post-synthesis verification interface |
| **H** | TrustScorer | Post-synthesis trust gate interface |

> For the full system design, agent responsibilities, signals, and decision policy, see `report/02_architecture.qmd`. For implementation refinements and fixes, see `report/03_reliability.qmd`.

## Prerequisites

- Notebook 02 completed: baseline retrievers (BM25, Dense, GraphRAG) built and stored in Google Drive
- Shared Drive folder `Adv_GenAI_FS26` accessible and paths configured in Section 2


## Section 1 — Installation

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("ADV_GENAI_RAG_REPO", "https://github.com/dhub100/advanced-genai-rag.git")
REPO_REF = os.environ.get("ADV_GENAI_RAG_REF", "improve-agent-a")
REPO_DIR = pathlib.Path("/content/advanced-genai-rag")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", "sentence-transformers", "faiss-cpu"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", str(REPO_DIR / "baseline" / "package")],
    check=True,
)

for name in list(sys.modules):
    if name == "rag" or name.startswith("rag."):
        del sys.modules[name]

sys.path.insert(0, str(REPO_DIR / "baseline" / "package" / "src"))

import rag
import rag.reliability
print(f"Installed advanced-genai-rag from {REPO_URL} @ {REPO_REF}")
print("rag loaded from:", rag.__file__)
print("rag.reliability module found.")


In [ ]:
import json
import time
import pathlib
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

## Section 2 — Google Drive & Configuration

Edit the `ROOT` path if your Drive layout differs from the default.

| Variable | Description |
|---|---|
| `PATH_BM25_PICKLE` | Serialised `QEBM25` object built in Notebook 02 |
| `PATH_DENSE_LOADER` | `load_dense_fixed.py` generated during corpus indexing |
| `PATH_GRAG_LOADER` | `load_graphrag.py` generated during graph construction |
| `PATH_QA` | Bilingual benchmark JSON (25 EN + 25 DE question-answer pairs) |
| `PATH_QRELS` | Folder of per-document relevance score JSON files |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ROOT = pathlib.Path("/content/drive/MyDrive/Adv_GenAI_FS26")
REPO_DIR = pathlib.Path("/content/advanced-genai-rag")

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"

PATH_QA_CANDIDATES = [
    REPO_DIR / "baseline/benchmark/benchmark_qa_bilingual.json",
    ROOT / "advanced-genai-rag/baseline/benchmark/benchmark_qa_bilingual.json",
]
PATH_QRELS_CANDIDATES = [
    REPO_DIR / "baseline/benchmark/score/fixed_size",
    ROOT / "advanced-genai-rag/baseline/benchmark/score/fixed_size",
]

PATH_QA = next((p for p in PATH_QA_CANDIDATES if p.exists()), PATH_QA_CANDIDATES[0])
PATH_QRELS = next((p for p in PATH_QRELS_CANDIDATES if p.exists()), PATH_QRELS_CANDIDATES[0])

for name, p in [("BM25", PATH_BM25_PICKLE), ("Dense loader", PATH_DENSE_LOADER),
                ("GraphRAG loader", PATH_GRAG_LOADER), ("QA benchmark", PATH_QA), ("Qrels", PATH_QRELS)]:
    status = "OK" if pathlib.Path(p).exists() else "NOT FOUND"
    print(f"  {status:>9}  {name}: {p}")

## Section 3 — Load Baseline System

Loads the pre-built retrieval components from Google Drive. Refer to Notebook 02 for detailed explanations of each component.

Two compatibility patches are applied before loading:
1. **`torch.is_available` patch** — the generated `load_dense_fixed.py` calls `torch.is_available()` instead of `torch.cuda.is_available()`. We register the alias before the file is imported.
2. **`__main__` pickle patch** — the BM25 object was pickled from a notebook where `BilingualBM25` and `QEBM25` were defined in `__main__`. We re-register the package classes under `__main__` so that `pickle.load` can resolve them.

In [ ]:
import __main__
import torch

# Patch 1: torch.is_available() alias
if not hasattr(torch, 'is_available'):
    torch.is_available = torch.cuda.is_available

# Patch 2: re-register pickled classes in __main__
from rag.retrieval.agents.bm25 import BilingualBM25, QEBM25
from rag.retrieval.translator import EnDeTranslator
__main__.BilingualBM25 = BilingualBM25
__main__.QEBM25 = QEBM25
__main__.EnDeTranslator = EnDeTranslator

from rag.retrieval.agents.bm25 import load_bm25_fixed_qe
from rag.retrieval.agents.dense import DenseRetriever, load_dense_fixed
from rag.retrieval.agents.graph import GraphAgent, load_graph_rag
from rag.retrieval.agents.answer_synthesizer import AnswerSynthesizerAgent
from rag.retrieval.orchestrator import Orchestrator

print("Loading BM25...")
bm25_fixed_qe = load_bm25_fixed_qe(bm25_pickle_path=str(PATH_BM25_PICKLE))
print("Loading Dense retriever...")
dense_fixed   = load_dense_fixed(dense_loader_path=str(PATH_DENSE_LOADER))
print("Loading GraphRAG...")
graph_rag     = load_graph_rag(loader_path=str(PATH_GRAG_LOADER))
print("Loading AnswerSynthesizer (Mistral-7B)...")
synthesizer   = AnswerSynthesizerAgent()

baseline_orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed,
    graph=graph_rag, synthesizer=synthesizer
)
print("✓ Baseline system ready.")

## Section 4 ? Evidence Sufficiency Estimation [Mechanism A]

This section instantiates Mechanism A through the `EvidenceSufficiencyChecker`, the pre-synthesis gate used by the reliable orchestrator. It evaluates whether the retrieved evidence is strong enough to justify answer generation.

Signals used here:

| Signal | Role | Threshold |
|---|---|---|
| `semantic_coverage` | topical match between query and retrieved chunks | ? 0.50 |
| `chunk_support_count` | number of individually supporting chunks | ? 3 |
| `aspect_coverage_ratio` | query-term coverage in retrieved text | ? 0.50 |
| `temporal_valid` | year-specific support check | boolean |
| `sufficiency_score` | composite evidence score | ? 0.45 |

The routing layer maps these signals to `answer`, `recover`, `clarify`, or `abstain`.

> Final design rationale for A is documented in `report/02_architecture.qmd`. The temporal-gap refinement and other implementation fixes are documented in `report/03_reliability.qmd`.


In [ ]:
import numpy as np
from rag.reliability.evidence_sufficiency import EvidenceSufficiencyChecker, EvidenceReport

# DenseRetriever wraps a ChromaDB store whose embedding function is accessible
# via store._embedding_function (a HuggingFaceEmbeddings instance).
def embed_fn(texts: list[str]) -> np.ndarray:
    return np.array(dense_fixed.store._embedding_function.embed_documents(texts))

checker = EvidenceSufficiencyChecker(embed_fn=embed_fn)
print("✓ EvidenceSufficiencyChecker ready.")

In [ ]:
def show_evidence_report(query: str, strategy: str = "confidence", top_k: int = 5):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=top_k)
    report = checker.check(query, result["documents"])

    print(f"Query  : {query}")
    print(f"{'─'*62}")
    print(f"  semantic_coverage     : {report.semantic_coverage:.3f}   (threshold ≥ 0.50)")
    print(f"  chunk_support_count   : {report.chunk_support_count}       (threshold ≥ 3)")
    print(f"  aspect_coverage_ratio : {report.aspect_coverage_ratio:.3f}   (threshold ≥ 0.50)")
    print(f"  temporal_valid        : {report.temporal_valid}")
    print(f"{'─'*62}")
    print(f"  sufficiency_score     : {report.sufficiency_score:.3f}   → sufficient = {report.sufficient}")
    print(f"  failure_type          : {report.failure_type}")
    if report.missing_aspects:
        print(f"  missing_aspects       : {report.missing_aspects}")
    print(f"  routing_decision      : {report.routing_decision}")
    print(f"  decision_reason       : {report.decision_reason}")
    return report

In [ ]:
print("=" * 62)
print("EXAMPLE 1 — likely sufficient evidence")
print("=" * 62)
_ = show_evidence_report("Who at ETH received ERC grants?")

print()
print("=" * 62)
print("EXAMPLE 2 — known baseline failure (hallucination case)")
print("=" * 62)
_ = show_evidence_report("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("EXAMPLE 3 — ambiguous query")
print("=" * 62)
_ = show_evidence_report("What research is ETH famous for?")

## Section 5 ? Recovery Agent [Mechanism G ? Adaptive Requirement]

The `RecoveryAgent` is the adaptive mechanism in the pipeline. When A reports insufficient evidence, G selects a corrective action that changes the next retrieval attempt instead of simply retrying unchanged.

Recovery actions used here include query rewriting, retrieval-strategy switching, and PRF-based expansion for lexical or cross-lingual mismatch.

> The full recovery policy and the PRF implementation correction are described in `report/02_architecture.qmd` and `report/03_reliability.qmd`.

### Query rewriting

LLM-based query rewriting uses the OpenAI API and requires a Colab Secret:
1. Click the **?? Secrets** icon in the left Colab sidebar
2. Add a secret named `OPENAI_API_KEY` with your key as the value
3. Toggle **Notebook access** on

Without a key, a rule-based fallback appends the missing key-terms to the original query.


In [ ]:
from rag.reliability.recovery import RecoveryAgent

openai_client = None
try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if openai_api_key:
        from openai import OpenAI
        openai_client = OpenAI(api_key=openai_api_key)
        print("✓ OpenAI client ready — LLM-based query rewriting enabled.")
    else:
        print("ℹ  OPENAI_API_KEY secret not set — rule-based query rewrite fallback will be used.")
except Exception:
    print("ℹ  Could not read Colab secrets — rule-based query rewrite fallback will be used.")

recovery_agent = RecoveryAgent(max_retries=2, openai_client=openai_client)
print("✓ RecoveryAgent ready.")

In [ ]:
# These queries are taken directly from the baseline failure analysis.
# All four fail consistently across every retrieval strategy in the baseline,
# making them reliable triggers for recovery behaviour.

RECOVERY_CASES = [
    {
        "query":    "who was president of ETH in 2003?",
        "gold":     "olaf kübler",
        "expected": "temporal_answer_not_supported — topic match without local year-to-role support; "
                    "query rewrite expected on attempt 1",
    },
    {
        "query":    "who were the rectors of ETH between 2017 and 2022?",
        "gold":     "sarah springman, günther dissertori",
        "expected": "missing_query_aspect — aspect terms absent from retrieved text; "
                    "query rewrite expected on attempt 1",
    },
    {
        "query":    "what is e-Sling?",
        "gold":     "4-seat electric airplane built by ETH students",
        "expected": "low_semantic_coverage — document exists in corpus in German only; "
                    "PRF expansion expected to improve cross-lingual match",
    },
    {
        "query":    "who was president of ETH in 2045?",
        "gold":     "N/A — future date, no document can exist in corpus",
        "expected": "temporal_mismatch — year 2045 is outside corpus range (1850–2030); "
                    "hard evidence mismatch prevents direct answering",
    },
]

def show_recovery_actions(query: str, strategy: str = "confidence"):
    def run_step(step_query: str, step_strategy: str, use_prf: bool):
        result = baseline_orchestrator.run(
            strategy=step_strategy,
            query=step_query,
            top_k=5,
            use_prf=use_prf,
        )
        report = checker.check(step_query, result["documents"])
        return result, report

    current_strategy = strategy
    current_query = query
    use_prf = False

    _, report = run_step(current_query, current_strategy, use_prf)

    print(f"Query        : {query}")
    print(f"sufficient   : {report.sufficient}  |  score={report.score:.3f}  |  failure='{report.failure_type}'")
    print(f"route        : {report.routing_decision}")
    print(f"reason       : {report.decision_reason}")
    print(f"temporal_valid: {report.temporal_valid}")

    if report.routing_decision in {"answer", "clarify", "abstain"}:
        print(f"  → Routing decision is '{report.routing_decision}'.")
        return

    for attempt in range(recovery_agent.max_retries + 1):
        action = recovery_agent.choose_action(report, attempt=attempt, current_strategy=current_strategy)
        print(f"  {action.trace_entry}")
        if action.type == "abstain":
            break

        if action.type == "rewrite_query":
            current_query = recovery_agent.rewrite_query(current_query, report.missing_aspects)
            print(f"    → Rewritten query: '{current_query}'")
        elif action.type == "switch_strategy":
            current_strategy = action.new_strategy
            print(f"    → New strategy: '{current_strategy}'")
        elif action.type == "prf_expansion":
            use_prf = True
            print("    → PRF enabled for next retrieval.")

        _, report = run_step(current_query, current_strategy, use_prf)
        print(f"    Updated     : sufficient={report.sufficient}  |  score={report.score:.3f}  |  failure='{report.failure_type}'")
        print(f"    Route       : {report.routing_decision}  |  strategy='{current_strategy}'  |  use_prf={use_prf}")
        print(f"    Reason      : {report.decision_reason}")

        if report.routing_decision in {"answer", "clarify", "abstain"}:
            print(f"    → Recovery loop stops with route '{report.routing_decision}'.")
            break


for case in RECOVERY_CASES:
    print("=" * 66)
    print(f"Expected : {case['expected']}")
    print(f"Gold     : {case['gold']}")
    print("=" * 66)
    show_recovery_actions(case["query"])
    print()

### Note on Mechanism A Refinement

Earlier versions of A could overestimate evidence quality for temporally invalid queries such as year-specific questions outside the corpus scope. The notebook now uses the refined checker with the `temporal_valid` signal included.

> The full ?observed limitation ? root cause ? fix? discussion is moved to `report/03_reliability.qmd`.


## Section 6 ? Abstention Gate [Mechanism E]

The `AbstentionGate` is the unified terminal state for reliability failures. It returns a structured abstention response rather than an unsupported answer or an uninformative empty output.

Entry paths used here:

| Path | Trigger | Method |
|---|---|---|
| Retrieval-side failure | hard floor or recovery exhausted | `abstain_evidence(...)` |
| Post-synthesis failure | trust below threshold | `abstain_low_trust(...)` |

> See `report/02_architecture.qmd` for the role of E in the two-gate policy.


In [ ]:
from rag.reliability.abstention import AbstentionMechanism, AbstentionResponse

abstainer = AbstentionMechanism()
print("✓ AbstentionMechanism ready.")

In [ ]:
def show_abstention_evidence(query: str):
    result = baseline_orchestrator.run(strategy="confidence", query=query, top_k=5)
    report = checker.check(query, result["documents"])
    trace  = [
        f"EvidenceCheck: score={report.score:.3f}, failure='{report.failure_type}'",
        "Recovery attempts exhausted."
    ]
    abstention = abstainer.abstain_evidence(query, report, trace)

    print(f"Query      : {query}")
    print(f"Trigger    : {abstention.trigger}")
    print(f"Reason     : {abstention.reason}")
    print(f"Missing    : {abstention.missing_aspects}")
    print(f"Confidence : {abstention.confidence}")
    print("Trace:")
    for e in abstention.trace:
        print(f"  {e}")


print("=" * 62)
print("ABSTENTION — evidence_failure path (A → G → E)")
print("=" * 62)
show_abstention_evidence("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("ABSTENTION — trust_failure path (B → H → E)")
print("=" * 62)
# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Replace the placeholder values with the real trust_score and
# groundedness_score produced by H and B respectively.
# ─────────────────────────────────────────────────────────────
abstention_trust = abstainer.abstain_low_trust(
    query="Who was president of ETH in 2003?",
    trust_score=0.0,          # placeholder — replace with H output
    groundedness_score=0.0,   # placeholder — replace with B output
    trace=["[H placeholder] trust_score below threshold"]
)
print(f"Trigger    : {abstention_trust.trigger}")
print(f"Reason     : {abstention_trust.reason}")
print(f"Trust      : {abstention_trust.trust_score}")

## Section 7 ? Groundedness Verification [Mechanism B]

> Placeholder interface for Mechanism B.

B is invoked after answer synthesis and is responsible for checking whether the generated answer is supported by the retrieved evidence. In this notebook, the component is left injectable so the pre-synthesis reliability path can be demonstrated independently.

Expected call:

```python
groundedness_score = grounder.check(query, answer, docs)
```

> See `report/02_architecture.qmd` for B?s architectural role.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PLACEHOLDER — Mechanism B (GroundednessVerifier)
# Implement your class here and assign it to groundedness_verifier
# ══════════════════════════════════════════════════════════════

# from rag.reliability.groundedness import GroundednessVerifier
# groundedness_verifier = GroundednessVerifier(...)

# Minimal stub — replace with real implementation
class GroundednessVerifierStub:
    """Stub — returns None so the orchestrator bypasses the post-synthesis gate."""
    def check(self, query: str, answer: str, docs: list) -> float:
        raise NotImplementedError("Mechanism B not yet implemented.")

# Leave as None until B is ready — the orchestrator skips the gate gracefully
groundedness_verifier = None
print("ℹ  groundedness_verifier = None  (Mechanism B placeholder)")

## Section 8 ? Trust Scoring [Mechanism H]

> Placeholder interface for Mechanism H.

H is the post-synthesis gate. It combines the evidence-side signal from A with the groundedness-side signal from B and decides whether the answer should be returned or routed to abstention.

Expected call:

```python
trust_score = trust_scorer.score(evidence_report, groundedness_score)
```

> See `report/02_architecture.qmd` for the full trust-gate design.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PLACEHOLDER — Mechanism H (TrustScorer)
# Implement your class here and assign it to trust_scorer
# ══════════════════════════════════════════════════════════════

# from rag.reliability.trust_scorer import TrustScorer
# trust_scorer = TrustScorer(...)

# Minimal stub — replace with real implementation
class TrustScorerStub:
    """Stub — raises NotImplementedError when called."""
    def score(self, evidence_report, groundedness_score) -> float:
        raise NotImplementedError("Mechanism H not yet implemented.")

# Leave as None until H is ready — the orchestrator skips the post-synthesis gate gracefully
trust_scorer = None
print("ℹ  trust_scorer = None  (Mechanism H placeholder)")

## Section 9 ? Reliable Orchestrator (A + G + E)

This section wires the implemented components into the `ReliableOrchestrator`. In the current notebook state, the pre-synthesis path (A ? G ? E) is fully runnable, while B and H remain injectable placeholders.

> For the full conditional routing policy, see `report/02_architecture.qmd`.


In [ ]:
from rag.reliability.reliable_orchestrator import ReliableOrchestrator, ReliableResponse

reliable_orchestrator = ReliableOrchestrator(
    orchestrator=baseline_orchestrator,
    embed_fn=embed_fn,
    max_retries=2,
    openai_client=openai_client,
    groundedness_verifier=groundedness_verifier,   # set in Section 7
    trust_scorer=trust_scorer,                     # set in Section 8
)
print("✓ ReliableOrchestrator ready.")

In [ ]:
def run_and_show(query: str, strategy: str = "confidence"):
    t0       = time.time()
    response = reliable_orchestrator.run(query=query, strategy=strategy, top_k=5)
    elapsed  = time.time() - t0

    print(f"Query      : {query}")
    print(f"Abstained  : {response.abstained}  |  Recoveries: {response.recovery_attempts}  |  {elapsed:.2f}s")

    if response.abstained:
        print(f"Trigger    : {response.abstention.trigger}")
        print(f"Reason     : {response.abstention.reason}")
    else:
        print(f"Answer     : {response.answer}")
        if response.evidence_report:
            print(f"Evidence   : score={response.evidence_report.score:.3f}")

    print("Trace (last 5 entries):")
    for entry in response.trace[-5:]:
        print(f"  {entry}")
    print()

In [ ]:
# Four representative cases drawn from the baseline failure analysis.
# The baseline returns hallucinated or "NOT FOUND" answers for all failure cases.
# Expected behaviour of the reliable system is stated for each.

TEST_CASES = [
    {
        "query":    "who at ETH received ERC grants?",
        "gold":     "tilman esslinger, ursula keller, and others",
        "note":     "Expected: recover if critical aspect 'grants' is missing from retrieved evidence",
    },
    {
        "query":    "who was president of ETH in 2003?",
        "gold":     "olaf kübler",
        "note":     "Baseline: hallucination ('Günther Huber'). "
                    "Expected: recover because year-to-role support is not explicit",
    },
    {
        "query":    "who were the rectors of ETH between 2017 and 2022?",
        "gold":     "sarah springman, günther dissertori",
        "note":     "Baseline: all strategies return NOT FOUND. "
                    "Expected: query rewrite recovery or abstention",
    },
    {
        "query":    "what is e-Sling?",
        "gold":     "4-seat electric airplane built by 20 ETH students",
        "note":     "Baseline: all strategies return NOT FOUND (doc is German-only). "
                    "Expected: PRF expansion recovery or abstention",
    },
    {
        "query":    "how do birds learn new songs?",
        "gold":     "incremental syllable adaptation, similar to child language acquisition",
        "note":     "Corpus gap — relevant document not in subsample. "
                    "Expected: abstention after recovery attempts exhausted",
    },
]

for case in TEST_CASES:
    print("=" * 66)
    print(f"Note: {case['note']}")
    print("=" * 66)
    run_and_show(case["query"])
    print(f"Gold: {case['gold']}")
    print()

## Section 10 ? Benchmark Evaluation

Runs the reliable orchestrator over the bilingual benchmark and reports abstention rate, recovery statistics, and per-query outcomes.


In [ ]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions, gold_answers = [], []
for x in qa_data:
    questions.append(x["question"])
    gold_answers.append(x["answer"])
    questions.append(x["question_de"])
    gold_answers.append(x["answer_de"])

print(f"Loaded {len(questions)} queries ({len(qa_data)} bilingual pairs).")

In [ ]:
reliable_results = []

for q, a in tqdm(zip(questions, gold_answers), total=len(questions), desc="Reliable pipeline"):
    t0 = time.time()
    r  = reliable_orchestrator.run(query=q, strategy="confidence", top_k=5)
    reliable_results.append({
        "query":      q,
        "gold":       a,
        "answer":     r.answer,
        "abstained":  r.abstained,
        "trigger":    r.abstention.trigger if r.abstained else None,
        "recoveries": r.recovery_attempts,
        "ev_score":   r.evidence_report.score if r.evidence_report else None,
        "latency":    time.time() - t0,
    })

rel_df = pd.DataFrame(reliable_results)
print("Done.")

In [ ]:
n            = len(reliable_results)
abstained_df = rel_df[rel_df["abstained"]]
recovered_df = rel_df[rel_df["recoveries"] > 0]
answered_df  = rel_df[~rel_df["abstained"]]

print(f"{'─'*40}")
print(f"  Total queries    : {n}")
print(f"  Answered         : {len(answered_df):3d}  ({len(answered_df)/n:.1%})")
print(f"  Abstained        : {len(abstained_df):3d}  ({len(abstained_df)/n:.1%})")
print(f"  Had recovery     : {len(recovered_df):3d}  ({len(recovered_df)/n:.1%})")
print(f"  Avg latency      : {rel_df['latency'].mean():.2f}s")
print(f"  Avg ev_score     : {rel_df['ev_score'].dropna().mean():.3f}")
print(f"{'─'*40}")

if len(abstained_df) > 0:
    print("\nAbstention trigger breakdown:")
    print(abstained_df["trigger"].value_counts().to_string())
    print("\nAbstained queries:")
    for _, row in abstained_df.iterrows():
        ev = f"{row['ev_score']:.2f}" if row['ev_score'] is not None else "N/A"
        print(f"  [{row['trigger']:20s}] ev={ev}  {row['query'][:60]}")

In [ ]:
if len(recovered_df) > 0:
    answered_after  = recovered_df[~recovered_df["abstained"]]
    abstained_after = recovered_df[recovered_df["abstained"]]

    print("Recovery outcomes:")
    print(f"  Answered after recovery  : {len(answered_after)}")
    print(f"  Abstained after recovery : {len(abstained_after)}")

    if len(answered_after) > 0:
        print("\nQueries where recovery produced an answer:")
        for _, row in answered_after.iterrows():
            print(f"  [recoveries={int(row['recoveries'])}] ev={row['ev_score']:.2f}  {row['query'][:60]}")
else:
    print("No recovery attempts triggered on this benchmark run.")